# Brand Model Trim Reference

이 notebook은 `cohort_grouping_input.csv`를 기반으로 `brand -> model_name -> trim_names` 구조의 JSON reference를 만든다.
이 JSON은 이후 비즈니스 로직에서 LLM에게 reference context로 넘기기 위한 용도다.


In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

DATA_CANDIDATES = [
    Path('data/cohort_grouping_input.csv'),
    Path('raw/models_cohort/cohort_generation/data/cohort_grouping_input.csv'),
]

for candidate in DATA_CANDIDATES:
    if candidate.exists():
        INPUT_PATH = candidate.resolve()
        break
else:
    raise FileNotFoundError('cohort_grouping_input.csv not found from the current notebook working directory.')

DATA_DIR = INPUT_PATH.parent
OUTPUT_PATH = DATA_DIR / 'brand_model_trim_reference.json'

INPUT_PATH, OUTPUT_PATH


(PosixPath('/Users/iwonbin/workspace/Study/boot/SKN28-1st-4team/data_collection/raw/models_cohort/cohort_generation/data/cohort_grouping_input.csv'),
 PosixPath('/Users/iwonbin/workspace/Study/boot/SKN28-1st-4team/data_collection/raw/models_cohort/cohort_generation/data/brand_model_trim_reference.json'))

In [2]:
df_raw = pd.read_csv(INPUT_PATH, dtype=str)
df = df_raw.apply(lambda col: col.map(lambda value: None if isinstance(value, str) and value.strip() == '' else value))
df.shape


(7754, 9)

## Build Flat Reference Table

우선 `brand + model_name` 기준으로 trim 목록을 모은 flat table을 만든다.
`-` 같은 placeholder trim은 제외한다.


In [3]:
def unique_trim_names(series: pd.Series) -> list[str]:
    values: list[str] = []
    for value in series.dropna().tolist():
        text = str(value).strip()
        if not text or text == '-':
            continue
        if text not in values:
            values.append(text)
    return values

reference_table = (
    df.groupby(['brand', 'model_name'], dropna=False)
    .agg(
        trim_names=('class_name', unique_trim_names),
        trim_count=('class_name', lambda s: len(unique_trim_names(s))),
    )
    .reset_index()
    .sort_values(['brand', 'model_name'])
    .reset_index(drop=True)
)

display(reference_table.head(30))
reference_table.shape


,brand,model_name,trim_names,trim_count
0,chevrolet,뉴 볼트 EV,[프리미어],1
1,chevrolet,더 넥스트 스파크,"[LS, LS 베이직, LT, LT 플러스, LTZ, 스페셜 에디션 패션, 스페셜 ...",12
2,chevrolet,더 넥스트 이쿼녹스,"[LS, LT, RS, 프리미어]",4
3,chevrolet,더 뉴 말리부,"[LS, LS 디럭스, LT, LT 디럭스, 레드라인 프리미어, 레드라인 프리미어 ...",15
4,chevrolet,더 뉴 스파크,"[LS, LS 베이직, LT, 레드픽 에디션, 마이핏 LT, 마이핏 에디션, 프리미...",8
5,chevrolet,더 뉴 아베오,"[L, LS, LT]",3
6,chevrolet,더 뉴 카마로 SS,[볼케이노 레드],1
7,chevrolet,더 뉴 트랙스,"[LS, LS 디럭스, LT, LT 디럭스, LT 코어, LTZ, 레드라인 LT 코...",11
8,chevrolet,더 뉴 트레일블레이저,"[LT, RS, 액티브, 프리미어]",4
9,chevrolet,리얼 뉴 콜로라도,"[익스트림, Z71-X, Z71-X 미드나잇, 익스트림-X]",4


(334, 4)

## Build Nested JSON Payload

최종 JSON은 `brand -> model_name -> trim_names` 구조를 가진다.


In [4]:
brand_model_trim_map: dict[str, dict[str, list[str]]] = {}
for row in reference_table.to_dict(orient="records"):
    brand = row["brand"]
    model_name = row["model_name"]
    trim_names = row["trim_names"]
    brand_model_trim_map.setdefault(brand, {})[model_name] = trim_names

reference_payload = {
    "metadata": {
        "source_file": INPUT_PATH.name,
        "brand_count": int(reference_table["brand"].nunique()),
        "model_count": int(len(reference_table)),
    },
    "brand_model_trim_map": brand_model_trim_map,
}

print(json.dumps(reference_payload["metadata"], ensure_ascii=False, indent=2))
print(json.dumps(dict(list(reference_payload["brand_model_trim_map"].items())[:1]), ensure_ascii=False, indent=2))


{
  "source_file": "cohort_grouping_input.csv",
  "brand_count": 5,
  "model_count": 334
}
{
  "chevrolet": {
    "뉴 볼트 EV": [
      "프리미어"
    ],
    "더 넥스트 스파크": [
      "LS",
      "LS 베이직",
      "LT",
      "LT 플러스",
      "LTZ",
      "스페셜 에디션 패션",
      "스페셜 에디션 퍼펙트 블랙",
      "에코 LS",
      "에코 LS 베이직",
      "에코 LT 플러스",
      "에코 LTZ",
      "베이직"
    ],
    "더 넥스트 이쿼녹스": [
      "LS",
      "LT",
      "RS",
      "프리미어"
    ],
    "더 뉴 말리부": [
      "LS",
      "LS 디럭스",
      "LT",
      "LT 디럭스",
      "레드라인 프리미어",
      "레드라인 프리미어 프라임 세이프티",
      "프리미어",
      "프리미어 퍼펙트 블랙 프라임 세이프티",
      "프리미어 퍼펙트 블랙 프리미어",
      "프리미어 프라임 세이프티",
      "LT 프리미엄",
      "LT 스페셜",
      "레드라인",
      "퍼펙트 블랙",
      "프리미어 스페셜"
    ],
    "더 뉴 스파크": [
      "LS",
      "LS 베이직",
      "LT",
      "레드픽 에디션",
      "마이핏 LT",
      "마이핏 에디션",
      "프리미어",
      "베이직"
    ],
    "더 뉴 아베오": [
      "L",
      "LS",
      "LT"
    ],
    "더 뉴 카마로 SS": [
      "볼케이노 레드"
    ],
    "더 뉴 트랙스": [

## Export JSON

`cohort_generation/data/brand_model_trim_reference.json`으로 저장한다.


In [5]:
OUTPUT_PATH.write_text(json.dumps(reference_payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
OUTPUT_PATH


PosixPath('/Users/iwonbin/workspace/Study/boot/SKN28-1st-4team/data_collection/raw/models_cohort/cohort_generation/data/brand_model_trim_reference.json')